In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [2]:
import os
from tqdm import tqdm
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from losses.final_lossv2 import InverseDistanceBoundaryDiceLoss
from models.cbam_nnunet import nnUNet3D_CBAM
from utils.unets_helper_functions import (
    set_seed,
    save_checkpoint,
    psv2_final_PatchDataset_cbam,
    compute_per_class_dice
    
)

In [3]:
set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [4]:
FOLD = 0

with open(f"../data/splits/fold_{FOLD}/train.txt") as f:
    train_cases = f.read().splitlines()

with open(f"../data/splits/fold_{FOLD}/val.txt") as f:
    val_cases = f.read().splitlines()

print("Train cases:", len(train_cases))
print("Val cases:", len(val_cases))


Train cases: 33
Val cases: 9


In [ ]:
PATCH_SIZE = 96
PATCHES_PER_CASE = 6
FOREGROUND_PROB = 0.25
GLOBAL_PROB = 0.4
NUM_WORKERS = 2
train_dataset = psv2_final_PatchDataset_cbam(
    train_cases,
    "../data/processed/imagesTr",
    "../data/processed/labelsTr",
    patches_per_case=PATCHES_PER_CASE,
    patch_size=PATCH_SIZE,
    foreground_prob=FOREGROUND_PROB,
    global_prob=GLOBAL_PROB, 
    augment=True
)

val_dataset = psv2_final_PatchDataset_cbam(
    val_cases,
    "../data/processed/imagesTr",
    "../data/processed/labelsTr",
    patches_per_case=2, 
    patch_size=PATCH_SIZE,
    foreground_prob=FOREGROUND_PROB,
    global_prob=GLOBAL_PROB,
    augment=False
)

train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

In [6]:
model = nnUNet3D_CBAM(
    in_channels=1,
    out_channels=7,
    base_filters=24 
).to(device)

print("cbam nnU-Net style model Initialized")


cbam nnU-Net style model Initialized


In [7]:
EPOCHS = 100

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
weights = torch.tensor([0.05, 1, 1.8, 2.2, 2.3, 2.3, 1]).to(device)
loss_fn = InverseDistanceBoundaryDiceLoss(
    epsilon=1e-5,
    lambda_weight=0.5,
    class_weights = weights
)
scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")


In [8]:
def train_one_epoch_cbam(model, loader, optimizer, scaler, loss_fn, device):

    model.train()
    total_loss = 0

    for images, labels, dist_maps in tqdm(loader):

        images = images.to(device)
        labels = labels.long().to(device)
        dist_maps = dist_maps.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast(device_type=device.type, enabled=device.type == "cuda"):

            outputs = model(images)

            # Deep supervision unpack
            out, ds2, ds3, ds4 = outputs


            loss_main = loss_fn(out, labels, dist_maps)
            loss_ds2  = loss_fn(ds2, labels, dist_maps)
            loss_ds3  = loss_fn(ds3, labels, dist_maps)
            loss_ds4  = loss_fn(ds4, labels, dist_maps)

            loss = (
                1.0 * loss_main +
                0.5 * loss_ds2 +
                0.25 * loss_ds3 +
                0.125 * loss_ds4
            )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    return total_loss / len(loader)

def validate_one_epoch_cbam(model, loader, loss_fn, device, num_classes=7):

    model.eval()
    total_loss = 0
    all_dices = []

    with torch.no_grad():
        for images, labels, dist_maps in loader:

            images = images.to(device)
            labels = labels.long().to(device)
            dist_maps = dist_maps.to(device)

            with torch.amp.autocast(device_type=device.type, enabled=device.type == "cuda"):

                outputs = model(images)
                out = outputs[0]

                loss = loss_fn(out, labels, dist_maps)

            total_loss += loss.item()

            batch_dice = compute_per_class_dice(out, labels, num_classes)
            all_dices.append(batch_dice)

    mean_loss = total_loss / len(loader)

    all_dices = np.array(all_dices)
    mean_dices = np.mean(all_dices, axis=0)

    return mean_loss, mean_dices

In [ ]:
def tta_predict(model, patch):
    preds = []

    for flip_dim in [None, 2, 3, 4]:
        if flip_dim is not None:
            patch_flip = torch.flip(patch, dims=[flip_dim])
        else:
            patch_flip = patch

        logits = model(patch_flip)[0]

        if flip_dim is not None:
            logits = torch.flip(logits, dims=[flip_dim])

        preds.append(torch.softmax(logits, dim=1))

    return torch.mean(torch.stack(preds), dim=0)

import nibabel as nib
def get_gaussian_weight_map(patch_size):
    import numpy as np

    ranges = [np.linspace(-1, 1, s) for s in patch_size]
    z, y, x = np.meshgrid(*ranges, indexing="ij")

    dist = z**2 + y**2 + x**2
    gaussian = np.exp(-dist / 0.3)

    gaussian = gaussian / np.max(gaussian)
    return torch.tensor(gaussian, dtype=torch.float32)


def final_sliding_window_cbam(model, image, patch_size=96, stride=48, device="cuda"):
    model.eval()

    image_t = torch.tensor(image, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
    _, _, D, H, W = image_t.shape
    num_classes = 7

    # CPU accumulation (safe for memory)
    output     = torch.zeros((1, num_classes, D, H, W), dtype=torch.float32)
    weight_map = torch.zeros_like(output)

    gaussian = get_gaussian_weight_map((patch_size, patch_size, patch_size))
    gaussian_gpu = gaussian.unsqueeze(0).unsqueeze(0).to(device)
    gaussian_cpu = gaussian.unsqueeze(0).unsqueeze(0)

    with torch.no_grad():
        z_steps = sorted(set(list(range(0, max(1, D - patch_size), stride)) + [max(0, D - patch_size)]))
        y_steps = sorted(set(list(range(0, max(1, H - patch_size), stride)) + [max(0, H - patch_size)]))
        x_steps = sorted(set(list(range(0, max(1, W - patch_size), stride)) + [max(0, W - patch_size)]))

        print(f"Volume: {D}x{H}x{W} | Patches: {len(z_steps)*len(y_steps)*len(x_steps)} | Stride: {stride}")

        for z in z_steps:
            for y in y_steps:
                for x in x_steps:

                    patch = image_t[:, :, z:z+patch_size, y:y+patch_size, x:x+patch_size]

                    # ---- Forward ----
                    logits = model(patch)[0]   # deep supervision → take main output
                    probs = tta_predict(model, patch)

                    # ---- Weight with Gaussian ----
                    probs = probs * gaussian_gpu

                    # ---- Move to CPU once ----
                    probs_cpu = probs.cpu()

                    output[:, :, z:z+patch_size, y:y+patch_size, x:x+patch_size] += probs_cpu
                    weight_map[:, :, z:z+patch_size, y:y+patch_size, x:x+patch_size] += gaussian_cpu

    # ---- Normalize ----
    output = output / weight_map.clamp(min=1e-6)

    # ---- Argmax ----
    pred = torch.argmax(output, dim=1).squeeze().numpy()

    return pred


def final_evaluate_full_volume_cbam(model, cases, images_dir, labels_dir,
                             patch_size=96, stride=48, device="cuda"):

    model.eval()
    all_dices = []

    for case in cases:

        image = nib.load(f"{images_dir}/{case}.nii.gz").get_fdata(dtype=np.float32)
        label = nib.load(f"{labels_dir}/{case}.nii.gz").get_fdata().astype(np.uint8)

        pred = final_sliding_window_cbam(
            model,
            image,
            patch_size=patch_size,
            stride=stride,
            device=device
        )

        print(f"\nCase: {case}")
        print("Pred labels:", np.unique(pred))
        print("GT labels:", np.unique(label))

        case_dices = []

        for cls in range(1, 7):  # ignore background

            pred_cls = (pred == cls)
            label_cls = (label == cls)

            intersection = np.sum(pred_cls & label_cls)
            union = np.sum(pred_cls) + np.sum(label_cls)

            # ---- SAFE Dice ----
            if union == 0:
                dice = 1.0  # both empty → perfect
            else:
                dice = (2 * intersection) / union

            case_dices.append(dice)

        all_dices.append(case_dices)

        print(f"Dice: {np.round(case_dices, 4)}")

    all_dices = np.array(all_dices)

    mean_dice = np.mean(all_dices, axis=0)
    std_dice  = np.std(all_dices, axis=0)

    print("\n===== FINAL RESULTS =====")
    print("Mean Dice:", np.round(mean_dice, 4))
    print("Std Dice :", np.round(std_dice, 4))

    return mean_dice, std_dice



In [10]:
def load_checkpoint(path, model, optimizer=None):
    checkpoint = torch.load(path, map_location=device)

    if "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])

        if optimizer is not None and "optimizer_state_dict" in checkpoint:
            optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

        epoch = checkpoint.get("epoch", -1)

        metric = checkpoint.get("metric", None)
        if metric is None:
            metric = checkpoint.get("best_combined", None)
        if metric is None:
            metric = checkpoint.get("best_val_loss", None)

    else:
        model.load_state_dict(checkpoint)
        epoch = -1
        metric = None

    return checkpoint, epoch, metric

In [11]:
best_val_loss = float("inf")
best_dice = 0.0
best_parotid = 0.0
best_combined = 0.0
start_epoch = 0

PROJECT_ROOT = os.path.abspath("..")
SAVE_DIR = os.path.join(PROJECT_ROOT, "experiments", "psv2")

os.makedirs(SAVE_DIR, exist_ok=True)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

print("Starting PSV2 training from scratch.")

Starting PSV2 training from scratch.


In [12]:
print(f"start_epoch: {start_epoch}")
print(f"save_dir: {SAVE_DIR}")
print(f"current_lr: {optimizer.param_groups[0]['lr']:.6f}")

start_epoch: 0
save_dir: c:\Users\Yeseswini\OneDrive\Desktop\dhanush_mini\mini-project-1\experiments\psv2
current_lr: 0.000200


In [13]:
print("Scratch setup complete. Start the training cell below to begin PSV2 training.")


Scratch setup complete. Start the training cell below to begin PSV2 training.


In [14]:
for epoch in range(start_epoch, EPOCHS):

    # ------------------ TRAIN ------------------
    train_loss = train_one_epoch_cbam(
        model,
        train_loader,
        optimizer,
        scaler,
        loss_fn,
        device
    )

    # ------------------ VALIDATE ------------------
    val_loss, val_dice = validate_one_epoch_cbam(
        model,
        val_loader,
        loss_fn,
        device
    )

    # Ignore background (class 0)
    mean_dice = val_dice[1:].mean()

    # Parotid (L=4, R=5)
    parotid_score = (val_dice[3] + val_dice[4]) / 2

    # 🔥 Combined metric (IMPORTANT)
    combined_score = 0.7 * mean_dice + 0.3 * parotid_score

    print(f"\nEpoch {epoch}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Per Class Dice: {val_dice}")
    print(f"Mean Dice: {mean_dice:.4f}")
    print(f"Parotid Dice (L,R): {val_dice[3]:.4f}, {val_dice[4]:.4f}")
    print(f"Combined Score: {combined_score:.4f}")

    # ------------------ SAVE BEST LOSS ------------------
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(
            model,
            optimizer,
            epoch,
            best_val_loss,
            os.path.join(SAVE_DIR, "psv2_best_loss_model.pth")
        )
        print("Saved Best Loss Model")

    # ------------------ SAVE BEST DICE ------------------
    if mean_dice > best_dice:
        best_dice = mean_dice
        save_checkpoint(
            model,
            optimizer,
            epoch,
            mean_dice,
            os.path.join(SAVE_DIR, "psv2_dice_model.pth")
        )
        print("Saved Best Dice Model")

    # ------------------ SAVE BEST PAROTID ------------------
    if parotid_score > best_parotid:
        best_parotid = parotid_score
        save_checkpoint(
            model,
            optimizer,
            epoch,
            parotid_score,
            os.path.join(SAVE_DIR, "psv2_parotid_model.pth")
        )
        print("Saved Best Parotid Model")

    # ------------------ SAVE BEST COMBINED ------------------
    if combined_score > best_combined:
        best_combined = combined_score
        save_checkpoint(
            model,
            optimizer,
            epoch,
            combined_score,
            os.path.join(SAVE_DIR, "psv2_combined_model.pth")
        )
        print("Saved Best Combined Model")

    # ------------------ SAVE LAST MODEL ------------------
    save_checkpoint(
        model,
        optimizer,
        epoch,
        val_loss,
        os.path.join(SAVE_DIR, "psv2_last_model.pth")
    )

    # ------------------ LR PRINT ------------------
    lr = optimizer.param_groups[0]['lr']
    print(f"LR: {lr:.6f}")

    # ------------------ SCHEDULER ------------------
    scheduler.step()

100%|██████████| 132/132 [04:45<00:00,  2.16s/it]



Epoch 0
Train Loss: 0.8961
Val Loss: 0.3863
Per Class Dice: [0.17807322 0.00256426 0.02155098 0.03190205 0.02603702 0.17089353]
Mean Dice: 0.0506
Parotid Dice (L,R): 0.0319, 0.0260
Combined Score: 0.0441
Saved Best Loss Model
Saved Best Dice Model
Saved Best Parotid Model
Saved Best Combined Model
LR: 0.000200


100%|██████████| 132/132 [04:31<00:00,  2.06s/it]



Epoch 1
Train Loss: 0.7205
Val Loss: 0.3577
Per Class Dice: [0.43498784 0.12769684 0.37256945 0.28684598 0.41328124 0.1434207 ]
Mean Dice: 0.2688
Parotid Dice (L,R): 0.2868, 0.4133
Combined Score: 0.2932
Saved Best Loss Model
Saved Best Dice Model
Saved Best Parotid Model
Saved Best Combined Model
LR: 0.000200


100%|██████████| 132/132 [04:08<00:00,  1.88s/it]


...


Epoch 97
Train Loss: 0.2623
Val Loss: 0.0738
Per Class Dice: [0.80656518 0.93358194 0.8504888  0.88347752 0.84896146 0.87632854]
Mean Dice: 0.8786
Parotid Dice (L,R): 0.8835, 0.8490
Combined Score: 0.8749
LR: 0.000000


100%|██████████| 132/132 [04:56<00:00,  2.25s/it]



Epoch 98
Train Loss: 0.2519
Val Loss: 0.1076
Per Class Dice: [0.76729582 0.87402443 0.79463475 0.76622604 0.79861701 0.84062077]
Mean Dice: 0.8148
Parotid Dice (L,R): 0.7662, 0.7986
Combined Score: 0.8051
LR: 0.000000


100%|██████████| 132/132 [06:28<00:00,  2.94s/it]



Epoch 99
Train Loss: 0.2780
Val Loss: 0.0994
Per Class Dice: [0.90189335 0.81892121 0.85707887 0.74497784 0.87435127 0.81866488]
Mean Dice: 0.8228
Parotid Dice (L,R): 0.7450, 0.8744
Combined Score: 0.8189
LR: 0.000000


In [15]:
mean_dice, std_dice = final_evaluate_full_volume_cbam(
    model,
    val_cases,
    "../data/processed/imagesTr",
    "../data/processed/labelsTr",
    device=device
)

Volume: 272x357x357 | Patches: 245 | Stride: 48

Case: case_38
Pred labels: [0 1 2 3 4 5 6]
GT labels: [0 1 2 3 4 5 6]
Dice: [0.9237 0.8363 0.8254 0.8447 0.8383 0.9069]
Volume: 243x383x383 | Patches: 245 | Stride: 48

Case: case_05
Pred labels: [0 1 2 3 4 5 6]
GT labels: [0 1 2 3 4 5 6]
Dice: [0.9279 0.7752 0.7446 0.798  0.831  0.8771]
Volume: 264x333x333 | Patches: 180 | Stride: 48


KeyboardInterrupt: 